# Deposit Attrition EDA — v10d · how much of the money is real

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

v10c settled the mechanism: deposit signals find attriters after the money has gone (97% drained
at the call), and where money is still on the book the combined model leads on AUC. It also left
three numbers that should not be quoted until tested:

| v10c result | why it is suspect |
|---|---|
| Ranking on current balance: **+68%** dollars | Inside the money-still-here set, ranking on `bar_now` can only gain by picking clients currently *above* their normal balance. The freeze assumption then credits the whole spike |
| Adding B clients: **11×** retained | $9.2m → $104.7m at a 1% save rate. Implausibly large against the size of the pool |
| `all_features` beats `payment_only` at α = 1 | $535m against $485m, from **67 clients** at roughly $8m each |

### A correction to my last read
I said the A-or-B reach of ~$6.1bn in seven months exceeded the whole pool's flow of ~$4bn. That
used v9's annualisation, which divided the pool by **31 months**. The pool only counts clients with
a balance reading 12 months before their event, so its events can span at most about 19 months —
which would put the seven-month flow nearer $6.5bn. The flag stands in a weaker form (a queue
reaching ~94% of all at-risk dollars is still implausible), but the comparison was rough. v10d
replaces it with an exact ceiling (§2) and measures the span directly (§4) — which also checks
v9's own $3.96bn annual figure.

### What v10d does
| § | |
|---|---|
| 1 | **Capped credit.** A save retains `min(bar_now, bar_med12)` — the lesser of what the client holds now and what it normally holds. Re-scores the headline and the ranking choice |
| 2 | **The attainable ceiling** — the most any queue could credit — and what share each queue captures. Replaces the pool comparison |
| 3 | **Concentration.** How many clients carry each queue's dollars; the top ten in the two flagged queues |
| 4 | **Does B money come back?** Balance 3–8 months after a B event, dollar-weighted; plus the pool's true event span |
| 5 | **Paired comparison with confidence intervals** — client bootstrap, whale jackknife, month-by-month |
| 6 | The headline, re-stated on capped credit, with intervals; how many calls |
| 7 | Scorecard |

**One discipline change.** v10c reported each feature set at its own best α, chosen on the test
months. Choosing on the test data flatters whichever set has the noisiest α curve. v10d compares
at **α = 0.5 and α = 1, fixed in advance**.

**Charts are saved, not shown.** Every chart is written to `OUT_DIR` and listed at the end; the
notebook shows tables only.

### Pre-registered predictions — checked in §7

| # | prediction | confidence |
|---|---|---|
| Q1 | Capping credit cuts the v10c headline (money still here, `all_features`, α = 1) by **less than 30%** | ~65% |
| Q2 | Under capped credit, ranking on current balance gains **less than 15%** over normal balance | ~70% |
| Q3 | Under capped credit, A-or-B reaches **less than 4×** the A-only dollars | ~60% |
| Q4 | `all_features` against `payment_only` at α = 1: the bootstrap 95% interval **includes zero** | ~65% |
| Q5 | `all_features` against `deposit_only` at α = 1: the interval **excludes zero** in favour of `all_features` | ~60% |
| Q6 | The top 10 clients carry **over half** the credited dollars in the v10c A-or-B queue | ~60% |
| Q7 | Dollar-weighted, **over 20%** of B-only event balances come back to half their normal level within 3–8 months | ~50% |
| Q8 | The pool's events span **fewer than 25 months**, so v9's ÷31 annualisation understated the flow | ~80% |

In [ ]:
# =====================================================================
# 0 · CONFIGURATION AND HELPERS — v10d
# =====================================================================
import warnings, time, math, hashlib
import numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display, HTML
warnings.filterwarnings("ignore")

HDFS_V6  = "hdfs://nameservice1/user/pk36814/attrition_v6"
HDFS_V9  = "hdfs://nameservice1/user/pk36814/attrition_v9"
def v6(n): return f"{HDFS_V6.rstrip('/')}/{n}"
def v9(n): return f"{HDFS_V9.rstrip('/')}/{n}"
V10C_DIR = Path(globals().get("V10C_DIR_OVERRIDE",
                              "/projects/DSI/sa15474/repos/pkg/eda/attrition_v10c"))
OUT_DIR  = Path(globals().get("OUT_DIR_OVERRIDE",
                              "/projects/DSI/sa15474/repos/pkg/eda/attrition_v10d"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED, MAX_ROWS = 20260909, 60
LABELS, CONFIGS = ["A_full_exit", "AB_any"], ["full", "defend"]
FS_ORDER = ["deposit_only", "payment_only", "both", "all_features"]
QUEUE_K = 1000
CAPACITY = [50, 100, 250, 500, 1000, 2500, 5000]
P_SAVE_GRID = [0.01, 0.05, 0.10, 0.20, 0.50]
RM_COST_PER_CALL = 250.0
ALPHA_COMMON = [0.5, 1.0]          # fixed in advance — never chosen on the test months
N_BOOT = 4000
JACK_K = [0, 1, 2, 3, 5, 10, 20]
TOP_N = 10
REBOUND_FRAC, REBOUND_WIN, NORM_REL = 0.5, (3, 8), -4
BAL_FLOOR = 1_000.0
V9_POOL_MONTHS = 31                # what v9 divided by to annualise
SHOW_IDS = False                   # tables show a short hash; full ids go to the CSVs only
RES, SAVED = {}, []

HAVE_MPL = True
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter
    plt.rcParams.update({
        "figure.dpi": 120, "font.size": 9, "axes.spines.top": False,
        "axes.spines.right": False, "axes.grid": True, "grid.color": "#E5E7EB",
        "grid.linewidth": .7, "axes.edgecolor": "#9AA1AC",
        "figure.facecolor": "white", "axes.facecolor": "white"})
except Exception as e:
    HAVE_MPL = False
    print(f"  matplotlib unavailable ({e}) — no charts will be saved")
ACC, ACC2, GOOD, WARN, GREY, INK = "#C1440E", "#4A6FA5", "#2F6F4E", "#B8860B", "#9AA1AC", "#16181D"
FSCOL = {"deposit_only": GREY, "payment_only": ACC2, "both": GOOD, "all_features": ACC}

def usd(v):
    try: v = float(v)
    except (TypeError, ValueError): return "—"
    if not np.isfinite(v): return "—"
    a, sg = abs(v), ("−" if v < 0 else "")
    if a >= 1e9: return f"{sg}${a/1e9:,.2f}bn"
    if a >= 1e6: return f"{sg}${a/1e6:,.1f}m"
    if a >= 1e3: return f"{sg}${a/1e3:,.0f}k"
    return f"{sg}${a:,.0f}"
def pctf(v, d=1):
    try:
        v = float(v)
        return f"{v:.{d}%}" if np.isfinite(v) else "—"
    except (TypeError, ValueError): return "—"
def cid(x):
    return str(x) if SHOW_IDS else "c_" + hashlib.sha1(str(x).encode()).hexdigest()[:6]
def _dec(s):
    from pyspark.sql import functions as F
    o = s
    for c, t in s.dtypes:
        if t.startswith("decimal"): o = o.withColumn(c, F.col(c).cast("double"))
    return o
def disp(o, title=None, n=None, save=None):
    n = MAX_ROWS if n is None else n
    out = o.copy() if isinstance(o, pd.DataFrame) else pd.DataFrame(o)
    if save: out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title:
        display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;"
                     f"margin:10px 0 2px;color:#111'>{title}</div>"))
    display(out.head(n)); return out
def kv(pairs, title=None, save=None):
    items = list(pairs.items()) if isinstance(pairs, dict) else list(pairs)
    labs = [k for k, _ in items]
    d = sorted({k for k in labs if labs.count(k) > 1})
    if d: raise ValueError(f"kv(): duplicate labels {d}")
    return disp(pd.DataFrame({"metric": labs, "value": [str(v) for _, v in items]}),
                title=title, n=len(items), save=save)

# ── charts: SAVED ONLY, never displayed ───────────────────────────────
def _save(fig, name, title, sub=None):
    if sub: fig.text(0.005, 0.965, sub, fontsize=8, color="#6B7280", va="top")
    fig.suptitle(title, fontsize=11, fontweight="bold", x=0.005, ha="left", y=1.0)
    fig.tight_layout(rect=[0, 0, 1, 0.93 if sub else 0.96])
    path = OUT_DIR / f"{name}.png"
    fig.savefig(path, format="png", bbox_inches="tight", dpi=130)
    plt.close(fig)
    SAVED.append(dict(chart=name, title=title, path=str(path)))
def bar_grouped(df, xlab, series, name, title, sub=None, ylab="", fmt=usd, colors=None,
                figsize=(9.5, 4.2), ylim=None):
    if not HAVE_MPL: return
    fig, ax = plt.subplots(figsize=figsize)
    n = len(series); w = 0.8/n; idx = np.arange(len(df))
    for i, s in enumerate(series):
        vals = df[s].to_numpy(dtype=float)
        b = ax.bar(idx + (i-(n-1)/2)*w, vals, w, label=str(s), color=(colors or {}).get(s))
        for r, v in zip(b, vals):
            if np.isfinite(v):
                ax.text(r.get_x()+r.get_width()/2, v, fmt(v), ha="center", va="bottom", fontsize=6.8)
    ax.set_xticks(idx); ax.set_xticklabels([str(i) for i in df.index], fontsize=8)
    ax.set_xlabel(xlab); ax.set_ylabel(ylab)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: fmt(v)))
    if ylim: ax.set_ylim(*ylim)
    else: ax.margins(y=.18)
    ax.legend(frameon=False, fontsize=8, ncol=min(n, 4))
    _save(fig, name, title, sub)
def heat(df, name, title, sub=None, fmt=usd, xlab="", ylab="", figsize=(9.5, 4.2)):
    if not HAVE_MPL: return
    fig, ax = plt.subplots(figsize=figsize)
    V = df.to_numpy(dtype=float)
    im = ax.imshow(V, cmap="OrRd", aspect="auto")
    ax.set_xticks(range(df.shape[1])); ax.set_xticklabels([str(c) for c in df.columns], fontsize=8)
    ax.set_yticks(range(df.shape[0])); ax.set_yticklabels([str(i) for i in df.index], fontsize=8)
    ax.set_xlabel(xlab); ax.set_ylabel(ylab); ax.grid(False)
    mx = np.nanmax(V) if np.isfinite(V).any() else 1.0
    for i in range(V.shape[0]):
        for j in range(V.shape[1]):
            if np.isfinite(V[i, j]):
                ax.text(j, i, fmt(V[i, j]), ha="center", va="center", fontsize=7.5,
                        color=("white" if V[i, j] > .62*mx else INK))
    cb = fig.colorbar(im, ax=ax, shrink=.85, pad=.015)
    cb.ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: fmt(v)))
    _save(fig, name, title, sub)

# ── the queue ─────────────────────────────────────────────────────────
def first_alerts(df, score_col, K, alpha=0.0, rank_col="bar_med12"):
    """score = p x rank_col^alpha, top K per month; a client's first alert is
    its one conversation. drop_duplicates keeps the whole first row."""
    s = pd.to_numeric(df[score_col], errors="coerce").to_numpy(float)
    if alpha > 0:
        b = np.maximum(pd.to_numeric(df[rank_col], errors="coerce").fillna(0.0).to_numpy(float), 1.0)
        s = s*np.power(b, alpha)
    d = df.assign(_s=s)
    d["_r"] = d.groupby("m_idx")["_s"].rank(ascending=False, method="first", na_option="bottom")
    fl = d[d._r <= K]
    first = fl.sort_values(["cust_pwr_id", "m_idx"]).drop_duplicates("cust_pwr_id", keep="first")
    return fl, first
def queue_tp(frame, fs, alpha, K=QUEUE_K, rank_col="bar_med12"):
    _, first = first_alerts(frame, f"p_{fs}", K, alpha, rank_col)
    return first, first[first.y == 1]
def savings_grid(frame, credit="bar_cap", alphas=ALPHA_COMMON, caps=CAPACITY,
                 rank_col="bar_med12", fsets=FS_ORDER):
    rows = []
    for fs in fsets:
        for alpha in alphas:
            for K in caps:
                first, tp = queue_tp(frame, fs, alpha, K, rank_col)
                reach = float(tp[credit].sum()); conv = int(len(first))
                for ps in P_SAVE_GRID:
                    saved, cost = reach*ps, conv*RM_COST_PER_CALL
                    rows.append(dict(feature_set=fs, alpha=alpha, k=K, p_save=ps, credit=credit,
                                     conversations=conv, tp_clients=int(len(tp)),
                                     dollars_reached=reach, saved_annualised=saved*ANNUALISE,
                                     rm_cost_annualised=cost*ANNUALISE,
                                     net_annualised=(saved-cost)*ANNUALISE,
                                     breakeven_p_save=cost/max(reach, 1.0)))
    return pd.DataFrame(rows)
def frontier(d):
    """Upper concave envelope of (conversations, dollars) from the origin.
    A costlier point holding no more money is dominated and dropped."""
    d = d.sort_values(["conversations", "dollars_reached"], ascending=[True, False])
    pts, best = [(0.0, 0.0, 0)], 0.0
    for r in d.itertuples():
        if float(r.dollars_reached) > best and float(r.conversations) > 0:
            pts.append((float(r.conversations), float(r.dollars_reached), int(r.k)))
            best = float(r.dollars_reached)
    hull = []
    for pt in pts:
        while len(hull) >= 2:
            (x0, y0, _), (x1, y1, _) = hull[-2], hull[-1]
            if (y1-y0)*(pt[0]-x1) <= (pt[1]-y1)*(x1-x0): hull.pop()
            else: break
        hull.append(pt)
    out = pd.DataFrame(hull, columns=["conversations", "dollars_reached", "k"])
    out["marginal_per_conversation"] = out.dollars_reached.diff()/out.conversations.diff()
    return out
print(f"helpers ready · charts {'saved to ' + str(OUT_DIR) if HAVE_MPL else 'OFF'}")


In [ ]:
# =====================================================================
# 0b · LOAD v10c's SCORED FRAMES — no refit                [OUTPUT BLOCK 1]
# =====================================================================
if "SC" in globals() and isinstance(globals()["SC"], dict) and len(globals()["SC"]) == 4:
    SCD = {k: v.copy() for k, v in globals()["SC"].items()}
    _src = "the live v10c kernel"
else:
    SCD, _missing = {}, []
    for d in LABELS:
        for c in CONFIGS:
            p = V10C_DIR / f"v10c_scored_{d}_{c}.parquet"
            if p.exists(): SCD[(d, c)] = pd.read_parquet(p)
            else: _missing.append(str(p))
    if _missing:
        raise FileNotFoundError("v10c scored frames not found — run v10c §4 first:\n" +
                                "\n".join(_missing))
    _src = str(V10C_DIR)
for k, fr in SCD.items():
    fr["bar_now"] = pd.to_numeric(fr.bar_now, errors="coerce").fillna(0.0).clip(lower=0)
    fr["bar_med12"] = pd.to_numeric(fr.bar_med12, errors="coerce").fillna(0.0).clip(lower=0)
    # THE CONSERVATIVE CREDIT: a save retains the lesser of what the client holds
    # now and what it normally holds. A transient spike is not a deposit.
    fr["bar_cap"] = np.minimum(fr.bar_now, fr.bar_med12)
ORIGINS = sorted(SCD[("A_full_exit", "full")].m_idx.unique().tolist())
ANNUALISE = 12.0/len(ORIGINS)
_d = SCD[("A_full_exit", "defend")]
kv([("source", _src),
    ("test months", f"{len(ORIGINS)} ({ORIGINS[0]}–{ORIGINS[-1]}) · annualise ×{ANNUALISE:.3f}"),
    *[(f"{d} · {c}", f"{len(fr):,} client-months · {int(fr.y.sum()):,} positive")
      for (d, c), fr in SCD.items()],
    ("money-still-here rows with no normal balance (credited $0 when capped)",
     f"{int((_d.bar_med12 <= 0).sum()):,}"),
    ("money-still-here positives currently ABOVE normal balance",
     pctf((_d.loc[_d.y == 1, "bar_now"] > _d.loc[_d.y == 1, "bar_med12"]).mean()))],
   title="0b &middot; Loaded", save="v10d_loaded")


In [ ]:
# =====================================================================
# 0c · LABELS — which event each client had (Spark, small) [OUTPUT BLOCK 2]
# =====================================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import StorageLevel
if "spark" not in globals():
    spark = (SparkSession.builder.appName("pkg_attrition_eda_v10d")
             .config("spark.sql.shuffle.partitions", "800")
             .config("spark.sql.execution.arrow.pyspark.enabled", "false")
             .enableHiveSupport().getOrCreate())
lab = spark.read.parquet(v6("labels_customer")).persist(StorageLevel.DISK_ONLY)
BAL = spark.read.parquet(v9("balance")).persist(StorageLevel.DISK_ONLY)
LABS = _dec(lab.select("cust_pwr_id", "q_A_full_exit", "q_B_bal_exit")).toPandas()
LABS = LABS.rename(columns={"q_A_full_exit": "qA", "q_B_bal_exit": "qB"})
LABS["qA"] = pd.to_numeric(LABS.qA, errors="coerce")
LABS["qB"] = pd.to_numeric(LABS.qB, errors="coerce")
kv([("clients with labels", f"{len(LABS):,}"),
    ("with an A event", f"{int(LABS.qA.notna().sum()):,}"),
    ("with a B event", f"{int(LABS.qB.notna().sum()):,}"),
    ("with B and never A", f"{int((LABS.qB.notna() & LABS.qA.isna()).sum()):,}")],
   title="0c &middot; Labels", save="v10d_labels")


## 1 · What the freeze assumption was crediting

v10c credited a save with `bar_now` — whatever the client held at the first alert. If that balance
was a spike, the save "retained" money that was never a normal deposit. The capped credit is
`min(bar_now, bar_med12)`: the lesser of the current and the normal balance.

- **1a** — for every queue, how much of the v10c credit sat above the client's normal balance (Q1)
- **1b** — the ranking choice re-scored on capped credit. If ranking on current balance still wins,
  the v10c +68% was real selection; if it collapses, it was spikes (Q2)

In [ ]:
# =====================================================================
# 1 · CAPPED CREDIT                                      [OUTPUT BLOCK 3]
# =====================================================================
def event_type(ids, event_m):
    L = LABS.assign(cust_pwr_id=LABS.cust_pwr_id.astype(str)).drop_duplicates("cust_pwr_id")
    L = L.set_index("cust_pwr_id")
    ids = pd.Series(ids).astype(str)
    qa = L.qA.reindex(ids).to_numpy(float); qb = L.qB.reindex(ids).to_numpy(float)
    ev = np.asarray(event_m, float)
    a, b = np.isclose(qa, ev), np.isclose(qb, ev)
    return np.where(a & b, "A and B same month",
           np.where(a, "A · closed", np.where(b, "B · drained, still open", "unmatched")))

rows = []
for (d, c), fr in SCD.items():
    for fs in FS_ORDER:
        _, tp = queue_tp(fr, fs, 1.0)
        now, cap = float(tp.bar_now.sum()), float(tp.bar_cap.sum())
        rows.append(dict(label=d, risk_set=c, feature_set=fs, departing_clients=len(tp),
                         credited_now=now, credited_capped=cap,
                         above_normal_share=(1 - cap/now) if now > 0 else np.nan))
CR = pd.DataFrame(rows)
_q = CR[(CR.label == "A_full_exit") & (CR.risk_set == "defend") & (CR.feature_set == "all_features")]
RES["cap_cut"] = float(_q.above_normal_share.iloc[0])
disp(CR.assign(credited_now=CR.credited_now.map(usd), credited_capped=CR.credited_capped.map(usd),
               above_normal_share=CR.above_normal_share.map(pctf)),
     title="1a &middot; <b>How much of each v10c credit sat above the client's normal balance</b> "
           "(&alpha; = 1, 1,000 alerts a month, ranked on normal balance as in v10c)",
     n=20, save="v10d_1a_capped_credit")
_p = CR[(CR.label == "A_full_exit") & (CR.risk_set == "defend")].set_index("feature_set").reindex(FS_ORDER)
bar_grouped(_p[["credited_now", "credited_capped"]], "feature set",
            ["credited_now", "credited_capped"], "v10d_1a_capped_credit",
            "1a · The v10c headline, credited at current balance and at the capped balance",
            "money still here · A_full_exit · α = 1 · 1,000 alerts/month · 7 test months",
            ylab="defendable $ reached", colors={"credited_now": GREY, "credited_capped": ACC})

# 1b — the ranking choice, re-scored
fr = SCD[("A_full_exit", "defend")]
rows = []
for fs in FS_ORDER:
    for rk in ["bar_med12", "bar_now", "bar_cap"]:
        _, tp = queue_tp(fr, fs, 1.0, rank_col=rk)
        rows.append(dict(feature_set=fs, rank_on=rk, departing_clients=len(tp),
                         credited_now=float(tp.bar_now.sum()),
                         credited_capped=float(tp.bar_cap.sum())))
RK = pd.DataFrame(rows)
Pn = RK.pivot(index="feature_set", columns="rank_on", values="credited_now").reindex(FS_ORDER)
Pc = RK.pivot(index="feature_set", columns="rank_on", values="credited_capped").reindex(FS_ORDER)
GAIN = pd.DataFrame({
    "uncapped · rank on current vs normal (v10c §8e)": Pn.bar_now/Pn.bar_med12 - 1,
    "CAPPED · rank on current vs normal": Pc.bar_now/Pc.bar_med12 - 1,
    "CAPPED · rank on capped vs normal": Pc.bar_cap/Pc.bar_med12 - 1})
RES["rank_now_gain_capped"] = float(GAIN.loc["all_features", "CAPPED · rank on current vs normal"])
disp(RK.assign(credited_now=RK.credited_now.map(usd), credited_capped=RK.credited_capped.map(usd)),
     title="1b &middot; <b>The ranking choice under both credits</b> — money still here, "
           "A_full_exit, &alpha; = 1", n=20, save="v10d_1b_rank_grid")
disp(GAIN.apply(lambda s: s.map(pctf)).reset_index(),
     title="1b &middot; <b>Gain over ranking on normal balance.</b> The first column reproduces "
           "v10c's +68%. If the second collapses, that gain was spikes being credited, not better "
           "selection", save="v10d_1b_rank_gain")
bar_grouped(Pc[["bar_med12", "bar_now", "bar_cap"]], "feature set",
            ["bar_med12", "bar_now", "bar_cap"], "v10d_1b_rank_capped",
            "1b · Capped dollars reached, by what the queue ranks on",
            "money still here · A_full_exit · α = 1 · credit = min(current, normal)",
            ylab="capped $ reached", colors={"bar_med12": GREY, "bar_now": ACC2, "bar_cap": ACC})


## 2 · The attainable ceiling

The most any queue could credit is, for each client who actually left, its best credited balance
across the months in which it was a positive — as if the RM had called at exactly the right time.
Every queue's reach is bounded by it by construction, so `capture` is a clean share. This replaces
the rough pool-flow comparison in my v10c read.

If the A-or-B ceiling is itself enormous, the 11× is a property of the **label**, not the model —
and 2b shows which kind of event carries it.

In [ ]:
# =====================================================================
# 2 · CEILING AND THE A-OR-B QUESTION                    [OUTPUT BLOCK 4]
# =====================================================================
def ceiling(fr, col):
    return fr[fr.y == 1].groupby("cust_pwr_id")[col].max()
rows = []
for (d, c), fr in SCD.items():
    cn, cc = ceiling(fr, "bar_now"), ceiling(fr, "bar_cap")
    for fs in FS_ORDER:
        _, tp = queue_tp(fr, fs, 1.0)
        rn, rc = float(tp.bar_now.sum()), float(tp.bar_cap.sum())
        assert rn <= cn.sum()*(1 + 1e-9) + 1 and rc <= cc.sum()*(1 + 1e-9) + 1, "reach exceeds ceiling"
        rows.append(dict(label=d, risk_set=c, feature_set=fs, clients_who_left=len(cn),
                         attainable_now=float(cn.sum()), attainable_capped=float(cc.sum()),
                         reached_now=rn, reached_capped=rc,
                         capture_now=rn/max(cn.sum(), 1), capture_capped=rc/max(cc.sum(), 1)))
CEIL = pd.DataFrame(rows)
disp(CEIL.assign(**{c: CEIL[c].map(usd) for c in ["attainable_now", "attainable_capped",
                                                 "reached_now", "reached_capped"]},
                 capture_now=CEIL.capture_now.map(pctf), capture_capped=CEIL.capture_capped.map(pctf)),
     title="2a &middot; <b>Attainable ceiling and capture</b> — &alpha; = 1, 1,000 alerts a month",
     n=20, save="v10d_2a_ceiling")
_g = lambda d, fs="all_features": CEIL[(CEIL.label == d) & (CEIL.risk_set == "defend") &
                                       (CEIL.feature_set == fs)].iloc[0]
_A, _B = _g("A_full_exit"), _g("AB_any")
RES["ab_ratio_capped"] = _B.reached_capped/max(_A.reached_capped, 1)
kv([("A only · reached (current / capped)", f"{usd(_A.reached_now)} / {usd(_A.reached_capped)}"),
    ("A or B · reached (current / capped)", f"{usd(_B.reached_now)} / {usd(_B.reached_capped)}"),
    ("A-or-B ÷ A-only, reached at current balance (v10c's ratio)",
     f"{_B.reached_now/max(_A.reached_now, 1):.1f}x"),
    ("A-or-B ÷ A-only, reached at CAPPED balance", f"{RES['ab_ratio_capped']:.1f}x"),
    ("A-or-B ÷ A-only, attainable ceiling (capped)",
     f"{_B.attainable_capped/max(_A.attainable_capped, 1):.1f}x"),
    ("clients who left — A only / A or B", f"{int(_A.clients_who_left):,} / {int(_B.clients_who_left):,}")],
   title="2b &middot; <b>Is the A-or-B jump the model or the label?</b> Money still here, "
         "all_features, &alpha; = 1", save="v10d_2b_ab_ratio")

# which kind of event carries the A-or-B dollars?
frab = SCD[("AB_any", "defend")]
pos = frab[frab.y == 1]
cl = (pos.groupby("cust_pwr_id").agg(event_m=("event_m", "first"), ceil_now=("bar_now", "max"),
                                     ceil_cap=("bar_cap", "max")).reset_index())
cl["event_type"] = event_type(cl.cust_pwr_id, cl.event_m)
_, tpab = queue_tp(frab, "all_features", 1.0)
cl = cl.merge(tpab[["cust_pwr_id", "bar_now", "bar_cap"]].rename(
        columns={"bar_now": "reached_now", "bar_cap": "reached_capped"}), on="cust_pwr_id", how="left")
cl[["reached_now", "reached_capped"]] = cl[["reached_now", "reached_capped"]].fillna(0.0)
COMP = (cl.groupby("event_type").agg(clients=("cust_pwr_id", "size"),
                                     attainable_capped=("ceil_cap", "sum"),
                                     reached_now=("reached_now", "sum"),
                                     reached_capped=("reached_capped", "sum")))
for c in ["attainable_capped", "reached_now", "reached_capped"]:
    COMP[f"{c}_share"] = COMP[c]/COMP[c].sum()
disp(COMP.reset_index().assign(**{c: COMP.reset_index()[c].map(usd) for c in
                                  ["attainable_capped", "reached_now", "reached_capped"]},
                               **{c: COMP.reset_index()[c].map(pctf) for c in
                                  ["attainable_capped_share", "reached_now_share", "reached_capped_share"]}),
     title="2c &middot; <b>What kind of event carries the A-or-B dollars.</b> A B event means the "
           "balance fell below 5% of normal for three months with accounts still open", n=10,
     save="v10d_2c_ab_composition")
bar_grouped(COMP[["reached_now", "reached_capped"]], "event type", ["reached_now", "reached_capped"],
            "v10d_2c_ab_composition", "2c · A-or-B queue dollars by the kind of event",
            "money still here · all_features · α = 1", ylab="$ reached",
            colors={"reached_now": GREY, "reached_capped": ACC})


In [ ]:
# =====================================================================
# 3 · CONCENTRATION — how many clients carry each queue  [OUTPUT BLOCK 5]
# =====================================================================
QUEUES = [
    ("A · all_features · α=1",                  ("A_full_exit", "defend"), "all_features", 1.0, "bar_med12"),
    ("A · payment_only · α=1",                  ("A_full_exit", "defend"), "payment_only", 1.0, "bar_med12"),
    ("A · deposit_only · α=1",                  ("A_full_exit", "defend"), "deposit_only", 1.0, "bar_med12"),
    ("A · all_features · α=0.5",                ("A_full_exit", "defend"), "all_features", 0.5, "bar_med12"),
    ("A · all_features · α=1 · rank on current", ("A_full_exit", "defend"), "all_features", 1.0, "bar_now"),
    ("A-or-B · all_features · α=1",             ("AB_any", "defend"), "all_features", 1.0, "bar_med12"),
]
def conc(v):
    v = np.sort(np.asarray(v, float))[::-1]; tot = v.sum()
    cs = np.cumsum(v)/tot if tot > 0 else np.zeros_like(v)
    top = lambda n: float(cs[min(n, len(v)) - 1]) if len(v) else np.nan
    return dict(departing_clients=len(v), total=tot, top1=top(1), top5=top(5), top10=top(10),
                top20=top(20), clients_for_half=int(np.searchsorted(cs, 0.5) + 1) if len(v) else 0), cs
rows, CURVES, TPS = [], {}, {}
for nm, key, fs, a, rk in QUEUES:
    _, tp = queue_tp(SCD[key], fs, a, rank_col=rk)
    TPS[nm] = tp
    for credit in ["bar_now", "bar_cap"]:
        s, cs = conc(tp[credit])
        rows.append(dict(queue=nm, credit=("current (v10c)" if credit == "bar_now" else "capped"), **s))
        CURVES[(nm, credit)] = cs
CONC = pd.DataFrame(rows)
RES["ab_top10_now"] = float(CONC[(CONC.queue == "A-or-B · all_features · α=1") &
                                 (CONC.credit == "current (v10c)")].top10.iloc[0])
disp(CONC.assign(total=CONC.total.map(usd), **{c: CONC[c].map(pctf) for c in
                                               ["top1", "top5", "top10", "top20"]}),
     title="3a &middot; <b>Concentration.</b> Share of each queue's credited dollars held by its "
           "largest departing clients, and how many clients make up half", n=20,
     save="v10d_3a_concentration")
if HAVE_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.0), sharey=True)
    for ax, credit, ttl in zip(axes, ["bar_now", "bar_cap"], ["current balance (v10c)", "capped"]):
        for (nm, *_r), col in zip(QUEUES, [ACC, ACC2, GREY, GOOD, WARN, INK]):
            cs = CURVES[(nm, credit)][:60]
            ax.plot(np.arange(1, len(cs)+1), cs, lw=2, color=col, label=nm)
        ax.set_title(ttl, fontsize=9.5); ax.set_xlabel("largest N departing clients")
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:.0%}"))
    axes[0].set_ylabel("share of the queue's credited dollars")
    axes[0].legend(frameon=False, fontsize=7)
    _save(fig, "v10d_3a_concentration", "3a · How few clients carry each queue",
          "money still here · 1,000 alerts/month · 7 test months")

def top_table(tp, n=TOP_N, with_type=False, save=None):
    t = tp.sort_values("bar_now", ascending=False).copy()
    tot = float(t.bar_now.sum())
    t["cum_share"] = t.bar_now.cumsum()/tot if tot > 0 else np.nan
    t = t.head(n)
    ratio = t.bar_now/t.bar_med12.where(t.bar_med12 > 0)
    out = pd.DataFrame({
        "rank": range(1, len(t)+1), "client": t.cust_pwr_id.map(cid).to_numpy(),
        "alert_month": t.m_idx.astype(int).to_numpy(), "event_month": t.event_m.astype(int).to_numpy(),
        "lead_months": (t.event_m - t.m_idx).astype(int).to_numpy(),
        "balance_now": t.bar_now.map(usd).to_numpy(), "normal_balance": t.bar_med12.map(usd).to_numpy(),
        "now_over_normal": ratio.map(lambda v: f"{v:.1f}x" if np.isfinite(v) else "—").to_numpy(),
        "credited_capped": t.bar_cap.map(usd).to_numpy(),
        "cum_share_of_queue": t.cum_share.map(pctf).to_numpy()})
    if with_type: out["event_type"] = event_type(t.cust_pwr_id, t.event_m)
    if save:   # the full ids go to disk only
        t.assign(event_type=event_type(t.cust_pwr_id, t.event_m) if with_type else "").to_csv(
            OUT_DIR / f"{save}_FULL_IDS.csv", index=False)
    return out
disp(top_table(TPS["A · all_features · α=1 · rank on current"], save="v10d_3b_top_rank_current"),
     title="3b &middot; <b>The ten largest credits when ranking on current balance</b> "
           "(v10c §8e). A high <code>now_over_normal</code> is a spike", save="v10d_3b_top_rank_current")
disp(top_table(TPS["A-or-B · all_features · α=1"], with_type=True, save="v10d_3c_top_ab"),
     title="3c &middot; <b>The ten largest credits in the A-or-B queue</b> (v10c §8g)",
     save="v10d_3c_top_ab")
TOPS = pd.concat([TPS[q].sort_values("bar_now", ascending=False).head(20).assign(queue=q)
                  for q in ["A · all_features · α=1 · rank on current", "A-or-B · all_features · α=1"]])
TOPS = TOPS[["cust_pwr_id", "queue", "m_idx", "event_m", "bar_now", "bar_med12"]].reset_index(drop=True)


## 4 · Does B money come back?

A B event is a balance below 5% of normal for three months with the accounts still open. For a
treasury client that can be a genuine exit — or cash moved into a sweep, a money-market vehicle or
another product, and back again. If the balance returns, the event was not attrition and the
A-or-B dollars in v10c §8g overstate what is at stake.

For every B client with enough history: the **normal balance four months before the event**
(before any version of the hold), against the **highest balance 3–8 months after it**. Closed
accounts (A) are measured the same way as a control; they should essentially never come back.

This cell also measures **the span of months the pool's events cover** (Q8) — which decides whether
v9's annualised pool figure was right.

In [ ]:
# =====================================================================
# 4 · REBOUND AFTER B EVENTS + THE POOL'S TRUE SPAN (Spark) [OUTPUT BLOCK 6]
# =====================================================================
att_A  = lab.filter(F.col("q_A_full_exit").isNotNull()).select(
    "cust_pwr_id", F.col("q_A_full_exit").alias("event_m"))
att_B  = (lab.filter(F.col("q_B_bal_exit").isNotNull() & F.col("q_A_full_exit").isNull())
          .select("cust_pwr_id", F.col("q_B_bal_exit").alias("event_m")))
att_BA = (lab.filter(F.col("q_B_bal_exit").isNotNull() & F.col("q_A_full_exit").isNotNull())
          .select("cust_pwr_id", F.col("q_B_bal_exit").alias("event_m")))

# (a) the pool's event span
def span(df, nm):
    r = (BAL.select("cust_pwr_id", "m_idx", "bal_med12").join(df, "cust_pwr_id")
         .withColumn("rel", F.col("m_idx") - F.col("event_m")).filter(F.col("rel") == -12)
         .agg(F.count(F.lit(1)).alias("n"), F.sum("bal_med12").alias("pool"),
              F.min("event_m").alias("first"), F.max("event_m").alias("last"),
              F.countDistinct("event_m").alias("distinct")).collect()[0])
    months = int(r["last"] - r["first"] + 1) if r["n"] else 0
    pool = float(r["pool"] or 0.0)
    return dict(cohort=nm, clients=int(r["n"]), pool=pool, first_event=int(r["first"] or 0),
                last_event=int(r["last"] or 0), event_months=months,
                distinct_event_months=int(r["distinct"] or 0),
                annualised_on_true_span=pool*12/max(months, 1),
                annualised_v9_method=pool*12/V9_POOL_MONTHS)
SPAN = pd.DataFrame([span(att_A, "A_full_exit"), span(att_B, "B_bal_exit only")])
RES["pool_months"] = int(SPAN.event_months.iloc[0])
disp(SPAN.assign(**{c: SPAN[c].map(usd) for c in ["pool", "annualised_on_true_span",
                                                 "annualised_v9_method"]}),
     title=f"4a &middot; <b>How many months the pool's events actually span.</b> v9 annualised by "
           f"dividing by {V9_POOL_MONTHS}; a client needs a balance 12 months before its event to "
           f"be in the pool, so the span is shorter", save="v10d_4a_pool_span")

# (b) rebound
def rebound(df, nm):
    j = (BAL.select("cust_pwr_id", "m_idx", "bal_now", "bal_med12")
         .join(F.broadcast(df), "cust_pwr_id")
         .withColumn("rel", F.col("m_idx") - F.col("event_m")))
    g = (j.groupBy("cust_pwr_id").agg(
            F.max(F.when(F.col("rel") == NORM_REL, F.col("bal_med12"))).alias("norm"),
            F.max(F.when(F.col("rel").between(*REBOUND_WIN), F.col("bal_now"))).alias("post_max"),
            F.sum(F.when(F.col("rel").between(*REBOUND_WIN), 1).otherwise(0)).alias("post_n")))
    p = _dec(g).toPandas()
    p = p[(p.post_n >= REBOUND_WIN[1] - REBOUND_WIN[0] + 1) & (p.norm > BAL_FLOOR)].copy()
    p["ratio"] = p.post_max/p.norm
    p["outcome"] = np.select([p.ratio >= REBOUND_FRAC, p.ratio >= 0.10],
                             ["came back (≥ half)", "partly back"], "gone")
    p["cohort"] = nm
    return p
RB = pd.concat([rebound(att_A, "A · closed (control)"), rebound(att_B, "B only · never closed"),
                rebound(att_BA, "B · closed later")], ignore_index=True)
RBS = (RB.groupby(["cohort", "outcome"]).agg(clients=("cust_pwr_id", "size"), dollars=("norm", "sum"))
       .reset_index())
RBS["dollar_share"] = RBS.dollars/RBS.groupby("cohort").dollars.transform("sum")
RBS["client_share"] = RBS.clients/RBS.groupby("cohort").clients.transform("sum")
_b = RBS[(RBS.cohort == "B only · never closed") & (RBS.outcome == "came back (≥ half)")]
RES["b_rebound_share"] = float(_b.dollar_share.iloc[0]) if len(_b) else 0.0
disp(RBS.assign(dollars=RBS.dollars.map(usd), dollar_share=RBS.dollar_share.map(pctf),
                client_share=RBS.client_share.map(pctf)),
     title=f"4b &middot; <b>Balance {REBOUND_WIN[0]}&ndash;{REBOUND_WIN[1]} months after the event, "
           f"against the normal balance {abs(NORM_REL)} months before it.</b> Dollar-weighted by "
           f"the normal balance; only clients with the full post-event window observed", n=20,
     save="v10d_4b_rebound")
if HAVE_MPL:
    piv = RBS.pivot(index="cohort", columns="outcome", values="dollar_share").fillna(0.0)
    piv = piv.reindex(columns=["gone", "partly back", "came back (≥ half)"], fill_value=0.0)
    fig, ax = plt.subplots(figsize=(9.5, 3.6))
    left = np.zeros(len(piv))
    for col, colr in zip(piv.columns, [GREY, WARN, ACC]):
        ax.barh(piv.index, piv[col], left=left, color=colr, label=col)
        for i, (l, v) in enumerate(zip(left, piv[col])):
            if v > .04: ax.text(l + v/2, i, f"{v:.0%}", ha="center", va="center", fontsize=8,
                                color="white")
        left += piv[col].to_numpy()
    ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:.0%}"))
    ax.legend(frameon=False, fontsize=8, ncol=3, loc="lower right")
    _save(fig, "v10d_4b_rebound", "4b · After the event, does the money come back?",
          "dollar-weighted by normal balance · closed accounts are the control")

# (c) the flagged clients' balances around their event
TOPS["cust_pwr_id"] = TOPS.cust_pwr_id.astype(str)
tops_s = spark.createDataFrame([(str(i), int(e)) for i, e in zip(TOPS.cust_pwr_id, TOPS.event_m)],
                               ["cust_pwr_id", "event_m"]).dropDuplicates(["cust_pwr_id"])
TRJ = _dec(BAL.select("cust_pwr_id", "m_idx", "bal_now").join(F.broadcast(tops_s), "cust_pwr_id")
           .withColumn("rel", F.col("m_idx") - F.col("event_m"))
           .filter(F.col("rel").between(-6, 8))).toPandas()
TRJ["cust_pwr_id"] = TRJ.cust_pwr_id.astype(str)
TW = TRJ.pivot_table(index="cust_pwr_id", columns="rel", values="bal_now")
pre = TW[[c for c in TW.columns if c < 0]].max(axis=1)
post = TW[[c for c in TW.columns if c >= REBOUND_WIN[0]]].max(axis=1)
T1 = (TOPS.drop_duplicates("cust_pwr_id").set_index("cust_pwr_id")
      .join(pd.DataFrame({"pre_max": pre, "post_max": post}), how="left"))
T1["event_type"] = event_type(T1.index, T1.event_m)
T1["back"] = np.where(T1.post_max >= REBOUND_FRAC*T1.pre_max, "came back", "gone")
T1.reset_index().to_csv(OUT_DIR / "v10d_4c_top_trajectories_FULL_IDS.csv", index=False)
_show_cols = [c for c in [-6, -3, -1, 0, 1, 3, 6, 8] if c in TW.columns]
_tw = TW[_show_cols].reindex(T1.index)
disp(pd.DataFrame({"queue": T1["queue"].to_numpy(), "client": [cid(i) for i in T1.index],
                   "event_type": T1.event_type.to_numpy(),
                   **{f"rel {c:+d}": _tw[c].map(usd).to_numpy() for c in _show_cols},
                   "outcome": T1.back.to_numpy()}),
     title="4c &middot; <b>Balances around the event for the largest credits in the two flagged "
           "queues</b> (months relative to the event)", n=40, save="v10d_4c_top_trajectories")


## 5 · Is the difference between feature sets real?

The queue decisions are held fixed; what varies is **which clients happened to leave**. Three views,
all on capped credit, money still here, A_full_exit, at α = 0.5 and α = 1 fixed in advance:

- **Client bootstrap.** Each client contributes its credited dollars under set X minus under set Y.
  Resampling clients gives a 95% interval for the difference and the probability X is ahead.
- **Whale jackknife.** The difference after dropping the 1, 2, 3, 5, 10, 20 clients with the largest
  contributions, and the number of clients **most favouring the leader** whose removal would flip
  the result — the fragility of the conclusion.
- **Month by month.** Each test month judged as a standalone queue; wins out of seven, and a
  t-interval on the monthly difference.

The bootstrap measures sampling luck in who left, not fitting noise in the models, so it is a lower
bound on the true uncertainty.

In [ ]:
# =====================================================================
# 5 · PAIRED COMPARISON                                  [OUTPUT BLOCK 7]
# =====================================================================
try:
    from scipy import stats as _st
    def tcrit(df): return float(_st.t.ppf(0.975, df)) if df > 0 else np.nan
except Exception:
    _TT = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306}
    def tcrit(df): return _TT.get(df, 1.96)

def contrib(frame, fs, alpha, credit="bar_cap"):
    _, tp = queue_tp(frame, fs, alpha)
    return tp.groupby("cust_pwr_id")[credit].sum().astype(float)

def paired(frame, fx, fy, alpha, credit="bar_cap", seed=SEED):
    cx, cy = contrib(frame, fx, alpha, credit), contrib(frame, fy, alpha, credit)
    ix = cx.index.union(cy.index)
    d = (cx.reindex(ix, fill_value=0.0) - cy.reindex(ix, fill_value=0.0)).to_numpy()
    D, n = float(d.sum()), len(d)
    rng = np.random.default_rng(seed)
    Db = np.empty(N_BOOT)
    for s in range(0, N_BOOT, 500):
        m = min(500, N_BOOT - s)
        Db[s:s+m] = d[rng.integers(0, n, size=(m, n))].sum(axis=1)
    lo, hi = np.percentile(Db, [2.5, 97.5])
    order = np.argsort(-np.abs(d))
    jack = {k: float(D - d[order[:k]].sum()) for k in JACK_K}
    lead = 1.0 if D >= 0 else -1.0
    rem = abs(D) - np.cumsum(np.sort(d*lead)[::-1])
    flip = int(np.argmax(rem <= 0) + 1) if (rem <= 0).any() else None
    md = []
    for T, g in frame.groupby("m_idx"):
        v = []
        for fs in (fx, fy):
            s_ = (pd.to_numeric(g[f"p_{fs}"], errors="coerce").to_numpy(float) *
                  np.power(np.maximum(g.bar_med12.to_numpy(float), 1.0), alpha))
            top = g.assign(_s=s_).nlargest(QUEUE_K, "_s")
            v.append(float(top.loc[top.y == 1, credit].sum()))
        md.append(v[0] - v[1])
    md = np.asarray(md); k = len(md)
    se = md.std(ddof=1)/np.sqrt(k) if k > 1 else np.nan
    return dict(alpha=alpha, comparison=f"{fx} − {fy}", clients_in_play=n, difference=D,
                ci_lo=float(lo), ci_hi=float(hi), p_first_ahead=float((Db > 0).mean()),
                at_1pct_annualised=D*0.01*ANNUALISE, ci_1pct_lo=lo*0.01*ANNUALISE,
                ci_1pct_hi=hi*0.01*ANNUALISE,
                clients_to_flip=("never — no subset reverses it" if flip is None else flip),
                months_first_wins=f"{int((md > 0).sum())} of {k}",
                month_ci_lo=float(md.mean() - tcrit(k-1)*se), month_ci_hi=float(md.mean() + tcrit(k-1)*se),
                **{f"drop_top_{j}": v for j, v in jack.items()}), Db

frA = SCD[("A_full_exit", "defend")]
PAIRS = [("all_features", "payment_only"), ("all_features", "deposit_only"),
         ("payment_only", "deposit_only")]
rows, BOOTS = [], {}
for a in ALPHA_COMMON:
    for fx, fy in PAIRS:
        r, Db = paired(frA, fx, fy, a); rows.append(r); BOOTS[(a, fx, fy)] = Db
PAIR = pd.DataFrame(rows)
_r = lambda a, fx, fy: PAIR[(PAIR.alpha == a) & (PAIR.comparison == f"{fx} − {fy}")].iloc[0]
RES["q4_ci"] = (float(_r(1.0, "all_features", "payment_only").ci_lo),
                float(_r(1.0, "all_features", "payment_only").ci_hi))
RES["q5_ci"] = (float(_r(1.0, "all_features", "deposit_only").ci_lo),
                float(_r(1.0, "all_features", "deposit_only").ci_hi))
_fmt = PAIR.assign(difference=PAIR.difference.map(usd),
                   ci_95=[f"{usd(a)} to {usd(b)}" for a, b in zip(PAIR.ci_lo, PAIR.ci_hi)],
                   p_first_ahead=PAIR.p_first_ahead.map(pctf),
                   at_1pct_annualised=PAIR.at_1pct_annualised.map(usd),
                   ci_at_1pct=[f"{usd(a)} to {usd(b)}" for a, b in zip(PAIR.ci_1pct_lo, PAIR.ci_1pct_hi)],
                   month_ci=[f"{usd(a)} to {usd(b)}" for a, b in zip(PAIR.month_ci_lo, PAIR.month_ci_hi)])
disp(_fmt[["alpha", "comparison", "clients_in_play", "difference", "ci_95", "p_first_ahead",
           "at_1pct_annualised", "ci_at_1pct", "clients_to_flip", "months_first_wins", "month_ci"]],
     title="5a &middot; <b>Paired differences in capped dollars reached</b> over the 7 test months. "
           "An interval that crosses zero means the ordering is not established. "
           "<code>clients_to_flip</code>: how many of the clients most favouring the leader would "
           "have to be absent for the result to reverse", n=10, save="v10d_5a_paired")
disp(PAIR[["alpha", "comparison"] + [f"drop_top_{j}" for j in JACK_K]]
     .assign(**{f"drop_top_{j}": PAIR[f"drop_top_{j}"].map(usd) for j in JACK_K}),
     title="5b &middot; <b>Whale jackknife</b> — the difference after dropping the N clients with "
           "the largest contributions either way", n=10, save="v10d_5b_jackknife")
if HAVE_MPL:
    fig, axes = plt.subplots(len(ALPHA_COMMON), len(PAIRS), figsize=(12, 6.4))
    for i, a in enumerate(ALPHA_COMMON):
        for j, (fx, fy) in enumerate(PAIRS):
            ax = axes[i, j]; Db = BOOTS[(a, fx, fy)]; r = _r(a, fx, fy)
            ax.hist(Db, bins=50, color=GREY, alpha=.9)
            ax.axvline(0, color=INK, ls=":", lw=1.2)
            ax.axvline(r.ci_lo, color=ACC, lw=1.2); ax.axvline(r.ci_hi, color=ACC, lw=1.2)
            ax.set_title(f"{fx} − {fy} · α={a}", fontsize=8.5)
            ax.xaxis.set_major_formatter(FuncFormatter(lambda v, p: usd(v)))
            ax.tick_params(axis="x", labelsize=7)
    _save(fig, "v10d_5a_bootstrap", "5a · Bootstrap distribution of each paired difference",
          "capped dollars reached over 7 test months · red = 95% interval · dotted = no difference")
    fig, ax = plt.subplots(figsize=(9.5, 4.0))
    for (a, fx, fy), col, ls in zip([(a, fx, fy) for a in ALPHA_COMMON for fx, fy in PAIRS],
                                    [ACC, ACC2, GOOD]*len(ALPHA_COMMON),
                                    ["-"]*len(PAIRS) + ["--"]*len(PAIRS)):
        r = _r(a, fx, fy)
        ax.plot(JACK_K, [r[f"drop_top_{j}"] for j in JACK_K], marker="o", ms=4, lw=1.8,
                color=col, ls=ls, label=f"{fx} − {fy} · α={a}")
    ax.axhline(0, color=INK, ls=":", lw=1)
    ax.set_xlabel("largest contributing clients dropped"); ax.set_ylabel("difference")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: usd(v)))
    ax.legend(frameon=False, fontsize=7)
    _save(fig, "v10d_5b_jackknife", "5b · How the differences survive dropping the largest clients",
          "a line crossing zero means a handful of clients decided the ordering")


In [ ]:
# =====================================================================
# 6 · THE HEADLINE ON CAPPED CREDIT + HOW MANY CALLS      [OUTPUT BLOCK 8]
# =====================================================================
SAV = savings_grid(frA, credit="bar_cap")
SAV.to_csv(OUT_DIR / "v10d_6_savings_capped.csv", index=False)
for a in ALPHA_COMMON:
    H = (SAV[(SAV.k == QUEUE_K) & (SAV.alpha == a)]
         .pivot(index="feature_set", columns="p_save", values="saved_annualised").reindex(FS_ORDER))
    disp(H.apply(lambda s: s.map(usd)).reset_index(),
         title=f"6a &middot; <b>Annualised retained on capped credit</b> — money still here, "
               f"A_full_exit, &alpha; = {a}, {QUEUE_K:,} alerts a month", save=f"v10d_6a_headline_a{a}")
    heat(H, f"v10d_6a_heat_a{a}", f"6a · Annualised retained on capped credit · α = {a}",
         f"money still here · A_full_exit · {QUEUE_K:,} alerts/month", xlab="RM save rate")
    bar_grouped(H.T, "RM save rate", FS_ORDER, f"v10d_6a_bar_a{a}",
                f"6a · Retained by feature set on capped credit · α = {a}",
                f"money still here · {QUEUE_K:,} alerts/month · annualised",
                ylab="annualised $ retained", colors=FSCOL)

rows = []
for a in ALPHA_COMMON:
    g = lambda fs: SAV[(SAV.feature_set == fs) & (SAV.k == QUEUE_K) & (SAV.alpha == a) &
                       (SAV.p_save == 0.01)].iloc[0]
    A_, P_, D_ = g("all_features"), g("payment_only"), g("deposit_only")
    rap, rad = _r(a, "all_features", "payment_only"), _r(a, "all_features", "deposit_only")
    rows += [(f"α={a} · all_features · departing clients reached", f"{int(A_.tp_clients):,}"),
             (f"α={a} · all_features · capped defendable dollars", usd(A_.dollars_reached)),
             (f"α={a} · all_features · retained at 1%, annualised", usd(A_.saved_annualised)),
             (f"α={a} · all_features · RM cost, annualised", usd(A_.rm_cost_annualised)),
             (f"α={a} · all_features · net at 1%, annualised", usd(A_.net_annualised)),
             (f"α={a} · all_features · break-even save rate", pctf(A_.breakeven_p_save, 3)),
             (f"α={a} · uplift over deposit_only at 1% (95% CI)",
              f"{usd(rad.at_1pct_annualised)} ({usd(rad.ci_1pct_lo)} to {usd(rad.ci_1pct_hi)})"),
             (f"α={a} · uplift over payment_only at 1% (95% CI)",
              f"{usd(rap.at_1pct_annualised)} ({usd(rap.ci_1pct_lo)} to {usd(rap.ci_1pct_hi)})")]
kv(rows, title="6b &middot; <b>The case, restated on capped credit, with intervals</b> — "
               "money still here, A_full_exit, 1,000 alerts a month", save="v10d_6b_case")

rows = []
for a in ALPHA_COMMON:
    for fs in FS_ORDER:
        d = SAV[(SAV.feature_set == fs) & (SAV.alpha == a) & (SAV.p_save == 0.10)][
            ["k", "conversations", "dollars_reached"]]
        f = frontier(d); f = f[f.k > 0].copy()
        f["feature_set"], f["alpha"] = fs, a
        f["kept_per_extra_call_1pct"] = f.marginal_per_conversation*0.01
        f["kept_per_extra_call_10pct"] = f.marginal_per_conversation*0.10
        rows.append(f)
FR = pd.concat(rows, ignore_index=True)
def _stop(fs, a, col):
    ok = FR[(FR.feature_set == fs) & (FR.alpha == a) & (FR[col] >= RM_COST_PER_CALL)]
    k = int(ok.k.max()) if len(ok) else 0
    return ("none" if k == 0 else (f"{k:,} — top of the grid" if k == max(CAPACITY) else f"{k:,}"))
disp(pd.DataFrame([dict(alpha=a, feature_set=fs,
                        frontier_points=int(((FR.feature_set == fs) & (FR.alpha == a)).sum()),
                        worthwhile_K_at_1pct=_stop(fs, a, "kept_per_extra_call_1pct"),
                        worthwhile_K_at_10pct=_stop(fs, a, "kept_per_extra_call_10pct"))
                   for a in ALPHA_COMMON for fs in FS_ORDER]),
     title=f"6c &middot; <b>How many alerts a month are worth making on capped credit</b> — the "
           f"largest K at which one more conversation still keeps more than {usd(RM_COST_PER_CALL)}",
     n=10, save="v10d_6c_stop")
if HAVE_MPL:
    fig, axes = plt.subplots(1, len(ALPHA_COMMON), figsize=(11, 4.0), sharey=True)
    for ax, a in zip(np.atleast_1d(axes), ALPHA_COMMON):
        for fs in FS_ORDER:
            f = FR[(FR.feature_set == fs) & (FR.alpha == a)].dropna(subset=["kept_per_extra_call_1pct"])
            ax.plot(f.conversations, f.kept_per_extra_call_1pct, marker="o", ms=4.5, lw=2,
                    color=FSCOL[fs], label=fs)
        ax.axhline(RM_COST_PER_CALL, color=INK, ls=":", lw=1.3)
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title(f"α = {a}", fontsize=9.5)
        ax.set_xlabel("distinct conversations over the test window")
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: usd(v)))
    np.atleast_1d(axes)[0].set_ylabel("$ kept by one more conversation, 1% save")
    np.atleast_1d(axes)[0].legend(frameon=False, fontsize=8)
    _save(fig, "v10d_6c_frontier", "6c · How many calls, on capped credit",
          f"dotted = {usd(RM_COST_PER_CALL)} cost of a call · stop where a curve crosses it")


In [ ]:
# =====================================================================
# 7 · SCORECARD                                          [OUTPUT BLOCK 9]
# =====================================================================
def _chk(key, test, fmt):
    if key not in RES: return "not evaluated", "—"
    try: return ("✓" if test(RES[key]) else "✗"), fmt(RES[key])
    except Exception as e: return f"error: {type(e).__name__}", "—"
_ci = lambda v: f"{usd(v[0])} to {usd(v[1])}"
PRED = [
    ("Q1", "Capping cuts the v10c headline by less than 30%", "~65%", "cap_cut",
     lambda v: v < 0.30, lambda v: f"cut {v:.1%}"),
    ("Q2", "Capped: ranking on current balance gains less than 15%", "~70%", "rank_now_gain_capped",
     lambda v: v < 0.15, lambda v: f"{v:+.1%}"),
    ("Q3", "Capped: A-or-B reaches less than 4x the A-only dollars", "~60%", "ab_ratio_capped",
     lambda v: v < 4.0, lambda v: f"{v:.1f}x"),
    ("Q4", "all_features − payment_only at α=1: 95% CI includes zero", "~65%", "q4_ci",
     lambda v: v[0] <= 0 <= v[1], _ci),
    ("Q5", "all_features − deposit_only at α=1: 95% CI above zero", "~60%", "q5_ci",
     lambda v: v[0] > 0, _ci),
    ("Q6", "Top 10 clients carry over half the v10c A-or-B queue", "~60%", "ab_top10_now",
     lambda v: v > 0.50, pctf),
    ("Q7", "Over 20% of B-only balances (dollar-weighted) come back", "~50%", "b_rebound_share",
     lambda v: v > 0.20, pctf),
    ("Q8", "The pool's events span fewer than 25 months", "~80%", "pool_months",
     lambda v: v < 25, lambda v: f"{v} months"),
]
SCORE = pd.DataFrame([dict(id=i, prediction=t, confidence=c, observed=_chk(k, tst, f)[1],
                           result=_chk(k, tst, f)[0]) for i, t, c, k, tst, f in PRED])
disp(SCORE, title="7 &middot; <b>Scorecard</b> — written in the introduction before this notebook "
     "ran; the result column is computed", n=10, save="v10d_7_scorecard")
print(f"  {int((SCORE.result == '✓').sum())} hit · {int((SCORE.result == '✗').sum())} missed · "
      f"{int((~SCORE.result.isin(['✓', '✗'])).sum())} not evaluated")


In [ ]:
# =====================================================================
# 8 · WHERE THE CHARTS ARE                               [OUTPUT BLOCK 10]
# =====================================================================
disp(pd.DataFrame(SAVED), title=f"8 &middot; <b>{len(SAVED)} charts saved</b> to "
     f"<code>{OUT_DIR}</code> — none are shown inline", n=60, save="v10d_8_chart_index")


---

## Reading this run

**§1 decides whether v10c's numbers stand.** If the capped cut is small (Q1), the headline was
honest and only needs a footnote. If ranking on current balance loses its advantage once capped
(Q2), v10c's +68% was spikes being credited and ranking stays on the normal balance.

**§2 and §4 decide what to do with B.** If the A-or-B ceiling is carried by B events and B money
comes back (Q7), B is catching treasury cash movements — sweeps and the like — not exits, and it
should stay out of the business case until the label is rebuilt to require the money to stay gone.
If B money does not come back, the drained-but-open population is real and the pool with it.

**§5 decides what can be said about the feature sets.** Where an interval crosses zero, the honest
statement is "no measurable difference in dollars", whatever the point estimate. Where it does not,
quote the interval, not the point. `clients_to_flip` is the plain-language version for a slide:
*this conclusion would reverse if N particular clients had not left*.

**§6 is the number to take forward** — capped credit, fixed α, with the interval on the uplift.

**§4a may change a figure already in circulation.** If the pool's events span ~19 months rather
than 31, v9's annualised pool of $3.96bn was understated by roughly 60%, and every "share of the
annual pool" statement built on it moves with it.